<a href="https://colab.research.google.com/github/avsnrakhila-2007/ML_lab_Programs/blob/main/ML_skill_05.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from google.colab import files

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    RocCurveDisplay
)

print("Please upload your placement CSV file...")

uploaded = files.upload()

filename = list(uploaded.keys())[0]

print("\nFile uploaded successfully:")
print(filename)

df = pd.read_csv(filename)

print("\n============================================================")
print("DATASET INFORMATION")
print("============================================================")

print("Dataset Shape:", df.shape)

print("\nFirst 5 rows:")
display(df.head())

print("\nColumn Names:")
print(df.columns.tolist())

print("\n============================================================")
print("MISSING VALUES")
print("============================================================")

print(df.isnull().sum())

NUM = [
    "SGPA_Sem1",
    "SGPA_Sem2",
    "SGPA_Sem3",
    "SGPA_Sem4",
    "SGPA_Sem5",
    "SGPA_Sem6",
    "SGPA_Sem7",
    "SGPA_Sem8",
    "CGPA",
    "AttendancePercent",
    "Internships",
    "Projects",
    "Workshops",
    "Certifications",
    "Publications",
    "AptitudeTestScore",
    "SoftSkillsRating",
    "CodingTestScore",
    "MockInterviewScore",
    "ExtraCurricular"
]

CAT = [
    "Gender",
    "City",
    "CollegeTier",
    "Stream",
    "Specialisation",
    "Hostel",
    "HistoryOfBacklogs",
    "CGPA_Tier"
]

TARGET = "PlacementStatus"

required_columns = NUM + CAT + [TARGET]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    print("\nERROR: These columns are missing from your dataset:")
    print(missing_columns)
    raise ValueError("Required columns are missing.")

print("\nAll required columns are present!")

X = df[NUM + CAT]

y = df[TARGET]

if y.dtype == "object":
    y = (
        y.astype(str)
        .str.strip()
        .str.lower()
        .map({
            "placed": 1,
            "not placed": 0,
            "yes": 1,
            "no": 0,
            "1": 1,
            "0": 0
        })
    )

y = pd.to_numeric(
    y,
    errors="coerce"
)

valid = y.notna()

X = X.loc[valid].copy()
y = y.loc[valid].astype(int)

print("\n============================================================")
print("TARGET DISTRIBUTION")
print("============================================================")

print(y.value_counts())

print("\nTarget percentage:")

print(
    (y.value_counts(normalize=True) * 100).round(2)
)

Xtr, Xte, ytr, yte = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("\n============================================================")
print("TRAIN-TEST SPLIT")
print("============================================================")

print("Training samples:", Xtr.shape[0])
print("Testing samples :", Xte.shape[0])

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

prep = ColumnTransformer([
    ("num", num_pipeline, NUM),
    ("cat", cat_pipeline, CAT)
])

clf = Pipeline([
    ("prep", prep),
    ("lr", LogisticRegression(max_iter=1000))
])

print("\n============================================================")
print("TRAINING LOGISTIC REGRESSION MODEL")
print("============================================================")

clf.fit(Xtr, ytr)

print("Model training completed successfully!")

proba = clf.predict_proba(Xte)[:, 1]

pred = clf.predict(Xte)

print("\n============================================================")
print("CLASSIFICATION REPORT")
print("============================================================")

print(
    classification_report(
        yte,
        pred,
        target_names=["Not placed", "Placed"]
    )
)

auc = roc_auc_score(
    yte,
    proba
)

print("\n============================================================")
print("ROC-AUC")
print("============================================================")

print("ROC-AUC:", round(auc, 4))

print("\n============================================================")
print("5-FOLD CROSS VALIDATION")
print("============================================================")

cv_scores = cross_val_score(
    clf,
    X,
    y,
    cv=5,
    scoring="accuracy"
)

print("Fold 1 Accuracy:", round(cv_scores[0], 4))
print("Fold 2 Accuracy:", round(cv_scores[1], 4))
print("Fold 3 Accuracy:", round(cv_scores[2], 4))
print("Fold 4 Accuracy:", round(cv_scores[3], 4))
print("Fold 5 Accuracy:", round(cv_scores[4], 4))

print(
    "\nMean 5-Fold CV Accuracy:",
    round(cv_scores.mean(), 4)
)

print("\n============================================================")
print("CONFUSION MATRIX")
print("============================================================")

ConfusionMatrixDisplay.from_estimator(
    clf,
    Xte,
    yte,
    cmap="Blues"
)

plt.title("Confusion Matrix - Placement Prediction")
plt.show()

print("\n============================================================")
print("ROC CURVE")
print("============================================================")

RocCurveDisplay.from_estimator(
    clf,
    Xte,
    yte
)

plt.plot(
    [0, 1],
    [0, 1],
    "k--"
)

plt.title("ROC Curve - Placement Prediction")
plt.show()

feature_names = (
    clf
    .named_steps["prep"]
    .get_feature_names_out()
)

coefficients = (
    clf
    .named_steps["lr"]
    .coef_[0]
)

odds_ratios = pd.Series(
    np.exp(coefficients),
    index=feature_names
)

odds_ratios = odds_ratios.sort_values(
    ascending=False
)

print("\n============================================================")
print("ODDS RATIOS")
print("============================================================")

print("\nOdds Ratio > 1 -> higher placement odds")
print("Odds Ratio < 1 -> lower placement odds")

print(
    odds_ratios.round(3)
)

print("\n============================================================")
print("TOP 15 FEATURES BY ODDS RATIO")
print("============================================================")

top_features = odds_ratios.head(15)

print(
    top_features.round(3)
)

print("\n============================================================")
print("LOWEST 15 FEATURES BY ODDS RATIO")
print("============================================================")

lowest_features = (
    odds_ratios
    .tail(15)
    .sort_values(ascending=True)
)

print(
    lowest_features.round(3)
)

coefficient_table = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients,
    "Odds_Ratio": np.exp(coefficients)
})

coefficient_table["Abs_Coefficient"] = (
    coefficient_table["Coefficient"].abs()
)

coefficient_table = coefficient_table.sort_values(
    "Abs_Coefficient",
    ascending=False
)

print("\n============================================================")
print("LOGISTIC REGRESSION COEFFICIENT TABLE")
print("============================================================")

display(
    coefficient_table[
        [
            "Feature",
            "Coefficient",
            "Odds_Ratio"
        ]
    ].round(4)
)

print("\n============================================================")
print("FINAL MODEL SUMMARY")
print("============================================================")

print("Dataset size          :", df.shape)
print("Training samples      :", len(Xtr))
print("Testing samples       :", len(Xte))
print(
    "5-Fold CV Accuracy    :",
    round(cv_scores.mean(), 4)
)
print(
    "ROC-AUC               :",
    round(auc, 4)
)

print("\nLogistic Regression Pipeline completed successfully!")